# Bab 4 (4.5 - 4.7): Membangun Data IoT Sintetis dengan Python

Buku: *IoT Terintegrasi dengan Machine Learning Data Analytics* (Onno W. Purbo), Bab 4 bagian 4.5 sampai 4.7, yang berada di antara Hands-on 8 dan Hands-on 9. Setiap sel kode sama dengan skrip pada folder `scripts/`.

Jalankan dari root repo atau dari folder `notebooks/` (sel pertama mengatur direktori kerja).

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(Path.cwd())

/home/sandbox/iot-ml-book


## 4.5.1 Contoh 1: suhu dan kelembaban harian (`scripts/generate_room_synthetic.py`)

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Waktu 1 hari, interval 10 menit
timestamps = pd.date_range(
    start="2026-01-01 00:00:00",
    end="2026-01-01 23:50:00",
    freq="10min"
)

hours = timestamps.hour + timestamps.minute / 60.0

# Pola suhu harian: rendah malam, naik siang
temperature = 24 + 4 * np.sin((hours - 6) / 24 * 2 * np.pi)
temperature += np.random.normal(0, 0.3, len(timestamps))

# kelembaban berlawanan dengan suhu
humidity = 70 - 10 * np.sin((hours - 6) / 24 * 2 * np.pi)
humidity += np.random.normal(0, 1.5, len(timestamps))

df = pd.DataFrame({
    "timestamp": timestamps,
    "device_id": "room01",
    "temperature_c": np.round(temperature, 2),
    "humidity_pct": np.round(humidity, 2)
})

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())

Data sintetik disimpan ke: data/synthetic/room_daily_synthetic.csv
            timestamp device_id  temperature_c  humidity_pct
0 2026-01-01 00:00:00    room01          19.61         81.83
1 2026-01-01 00:10:00    room01          20.09         82.04
2 2026-01-01 00:20:00    room01          20.32         79.58
3 2026-01-01 00:30:00    room01          19.81         78.59
4 2026-01-01 00:40:00    room01          19.73         78.17


## 4.5.2 Contoh 2: konsumsi energi dengan pola jam sibuk (`scripts/generate_energy_synthetic.py`)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

timestamps = pd.date_range(
    start="2026-01-01 00:00:00",
    end="2026-01-01 23:50:00",
    freq="10min"
)

power = []

for ts in timestamps:
    hour = ts.hour

    if 0 <= hour < 6:
        base = 120
    elif 6 <= hour < 8:
        base = 180
    elif 8 <= hour < 12:
        base = 350
    elif 12 <= hour < 14:
        base = 450
    elif 14 <= hour < 18:
        base = 320
    else:
        base = 170

    value = base + np.random.normal(0, 15)
    power.append(value)

df = pd.DataFrame({
    "timestamp": timestamps,
    "device_id": "lab-energy-01",
    "power_watt": np.round(power, 2)
})

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data energi sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())

Data energi sintetik disimpan ke: data/synthetic/energy_daily_synthetic.csv
            timestamp      device_id  power_watt
0 2026-01-01 00:00:00  lab-energy-01      113.29
1 2026-01-01 00:10:00  lab-energy-01      133.95
2 2026-01-01 00:20:00  lab-energy-01      105.78
3 2026-01-01 00:30:00  lab-energy-01      133.11
4 2026-01-01 00:40:00  lab-energy-01      128.27


## 4.5.3 Contoh 3: getaran mesin normal vs fault (`scripts/generate_vibration_synthetic.py`)

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_PATH = Path("data/synthetic/vibration_machine_synthetic.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

n = 1000
t = np.arange(n)

# Sinyal normal
normal_signal = 0.8 * np.sin(2 * np.pi * 0.03 * t) + np.random.normal(0, 0.1, n)

# Sinyal fault
fault_signal = 1.6 * np.sin(2 * np.pi * 0.03 * t) + np.random.normal(0, 0.25, n)

df_normal = pd.DataFrame({
    "sample_id": range(n),
    "machine_id": "motor01",
    "vibration": np.round(normal_signal, 4),
    "label": "normal"
})

df_fault = pd.DataFrame({
    "sample_id": range(n, 2*n),
    "machine_id": "motor01",
    "vibration": np.round(fault_signal, 4),
    "label": "fault"
})

df = pd.concat([df_normal, df_fault], ignore_index=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Data getaran sintetik disimpan ke: {OUTPUT_PATH}")
print(df.head())
print(df.tail())

Data getaran sintetik disimpan ke: data/synthetic/vibration_machine_synthetic.csv
   sample_id machine_id  vibration   label
0          0    motor01    -0.0227  normal
1          1    motor01    -0.0266  normal
2          2    motor01     0.3078  normal
3          3    motor01     0.4015  normal
4          4    motor01     0.4134  normal
      sample_id machine_id  vibration  label
1995       1995    motor01    -0.9835  fault
1996       1996    motor01    -1.0727  fault
1997       1997    motor01    -0.5631  fault
1998       1998    motor01    -0.4766  fault
1999       1999    motor01     0.0083  fault


## 4.5.4 Menambahkan noise pada data sensor

In [5]:
import numpy as np

clean_temp = np.array([25.0, 25.2, 25.4, 25.3, 25.5])
noisy_temp = clean_temp + np.random.normal(0, 0.2, len(clean_temp))

print("Data asli :", clean_temp)
print("Data noisy:", np.round(noisy_temp, 2))

Data asli : [25.  25.2 25.4 25.3 25.5]
Data noisy: [24.87 24.96 25.4  25.33 25.8 ]


## 4.5.5 Missing value, drift, dan outlier buatan (`scripts/inject_data_issues.py`)

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/room_daily_synthetic_corrupted.csv")

df = pd.read_csv(INPUT_PATH)

# 1. Missing value acak
missing_idx = np.random.choice(df.index, size=8, replace=False)
df.loc[missing_idx, "temperature_c"] = np.nan

# 2. Drift bertahap pada suhu
df["temperature_c"] = df["temperature_c"] + np.linspace(0, 1.2, len(df))

# 3. Outlier buatan
outlier_idx = np.random.choice(df.index, size=3, replace=False)
df.loc[outlier_idx, "temperature_c"] = df.loc[outlier_idx, "temperature_c"] + 8

df.to_csv(OUTPUT_PATH, index=False)
print(f"Data dengan missing value, drift, dan outlier disimpan ke: {OUTPUT_PATH}")
print("Index missing:", missing_idx)
print("Index outlier:", outlier_idx)

Data dengan missing value, drift, dan outlier disimpan ke: data/synthetic/room_daily_synthetic_corrupted.csv
Index missing: [108  96  21 137 113  60 129  92]
Index outlier: [  4  36 104]


## 4.5.6 Contoh 1: label nyaman / tidak nyaman (`scripts/label_room_comfort_daily.py`)

In [7]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/room_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/room_daily_labeled.csv")

df = pd.read_csv(INPUT_PATH)

df["comfort_label"] = df.apply(
    lambda row: "nyaman"
    if (24 <= row["temperature_c"] <= 28 and 45 <= row["humidity_pct"] <= 65)
    else "tidak_nyaman",
    axis=1
)

df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp device_id  temperature_c  humidity_pct comfort_label
0  2026-01-01 00:00:00    room01          19.61         81.83  tidak_nyaman
1  2026-01-01 00:10:00    room01          20.09         82.04  tidak_nyaman
2  2026-01-01 00:20:00    room01          20.32         79.58  tidak_nyaman
3  2026-01-01 00:30:00    room01          19.81         78.59  tidak_nyaman
4  2026-01-01 00:40:00    room01          19.73         78.17  tidak_nyaman


## 4.5.6 Contoh 2: label normal / warning / fault (`scripts/label_energy_status_daily.py`)

In [8]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/energy_daily_labeled.csv")

df = pd.read_csv(INPUT_PATH)

def label_energy(power):
    if power < 250:
        return "normal"
    elif power < 400:
        return "warning"
    return "fault"

df["status_label"] = df["power_watt"].apply(label_energy)
df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp      device_id  power_watt status_label
0  2026-01-01 00:00:00  lab-energy-01      113.29       normal
1  2026-01-01 00:10:00  lab-energy-01      133.95       normal
2  2026-01-01 00:20:00  lab-energy-01      105.78       normal
3  2026-01-01 00:30:00  lab-energy-01      133.11       normal
4  2026-01-01 00:40:00  lab-energy-01      128.27       normal


## 4.5.6 Contoh 3: label hemat / boros (`scripts/label_energy_efficiency_daily.py`)

In [9]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data/synthetic/energy_daily_synthetic.csv")
OUTPUT_PATH = Path("data/synthetic/energy_daily_efficiency.csv")

df = pd.read_csv(INPUT_PATH)

mean_power = df["power_watt"].mean()

df["efficiency_label"] = df["power_watt"].apply(
    lambda x: "hemat" if x <= mean_power else "boros"
)

df.to_csv(OUTPUT_PATH, index=False)
print(df.head())

             timestamp      device_id  power_watt efficiency_label
0  2026-01-01 00:00:00  lab-energy-01      113.29            hemat
1  2026-01-01 00:10:00  lab-energy-01      133.95            hemat
2  2026-01-01 00:20:00  lab-energy-01      105.78            hemat
3  2026-01-01 00:30:00  lab-energy-01      133.11            hemat
4  2026-01-01 00:40:00  lab-energy-01      128.27            hemat


## 4.7 Validasi sederhana (`scripts/validate_synthetic_room.py`)

In [10]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/synthetic/room_daily_synthetic.csv")
df = pd.read_csv(DATA_PATH)

print("=== Statistik dasar ===")
print(df.describe())

print("\n=== Korelasi ===")
print(df[["temperature_c", "humidity_pct"]].corr())

print("\n=== Cek nilai ekstrem ===")
print("Suhu min:", df["temperature_c"].min())
print("Suhu max:", df["temperature_c"].max())
print("Humidity min:", df["humidity_pct"].min())
print("Humidity max:", df["humidity_pct"].max())

=== Statistik dasar ===
       temperature_c  humidity_pct
count     144.000000    144.000000
mean       24.046528     69.932986
std         2.798517      7.312387
min        19.610000     58.710000
25%        21.307500     62.840000
50%        24.105000     69.635000
75%        26.862500     76.800000
max        28.400000     82.070000

=== Korelasi ===
               temperature_c  humidity_pct
temperature_c       1.000000     -0.968695
humidity_pct       -0.968695      1.000000

=== Cek nilai ekstrem ===
Suhu min: 19.61
Suhu max: 28.4
Humidity min: 58.71
Humidity max: 82.07
